# 04 — Model Training and Temporal Validation

This notebook develops and evaluates the three machine-learning approaches used in the study:

- LightGBM;
- CatBoost;
- multilayer perceptron (MLP).

Model development uses only the 2010–2022 development period.

Two chronological cross-validation strategies are compared:

- **sliding window**, representing adaptation to more recent observations;
- **expanding window**, retaining all historical information available up to each validation period.

The independent year 2023 remains isolated from model development and will be used only for final evaluation.

In [18]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd

from copd_forecasting.models import (
    OPTIMAL_PARAMS,
    evaluate_temporal_cv,
)

from copd_forecasting.config import (
    PROCESSED_DATA_FILE,
    FEATURE_SELECTION_FILE,
    DATE_COLUMN,
    TEST_YEAR,
    CV_RESULTS_FILE,
)

from copd_forecasting.features import (
    add_temporal_features,
    create_binary_target,
    build_subset_a,
    build_subset_b,
)

from copd_forecasting.selection import (
    prepare_feature_subset,
)

from copd_forecasting.validation import (
    get_year_based_splits,
    summarize_temporal_splits,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Reconstruct modelling datasets

The cleaned dataset is used to reconstruct the two predictor subsets.

The correlation filter and Min-Max scaler are fitted using the development period only, while the final feature lists are loaded from the results produced in the previous notebook.

In [2]:
df = pd.read_excel(
    PROCESSED_DATA_FILE,
    parse_dates=[DATE_COLUMN],
)

df = (
    df
    .sort_values(DATE_COLUMN)
    .reset_index(drop=True)
)

df = add_temporal_features(df)

y = create_binary_target(df)

subset_a = build_subset_a(df)
subset_b = build_subset_b(df)

In [3]:
development_mask = (
    df[DATE_COLUMN].dt.year
    < TEST_YEAR
).to_numpy()

test_mask = (
    df[DATE_COLUMN].dt.year
    == TEST_YEAR
).to_numpy()

y_development = (
    y[development_mask]
    .to_numpy()
)

y_test = (
    y[test_mask]
    .to_numpy()
)

development_dates = (
    df.loc[
        development_mask,
        DATE_COLUMN,
    ]
    .reset_index(drop=True)
)

print(
    f"Development samples: "
    f"{development_mask.sum():,}"
)

print(
    f"Final test samples: "
    f"{test_mask.sum():,}"
)

Development samples: 4,741
Final test samples: 365


In [4]:
prepared_a = prepare_feature_subset(
    subset_a,
    development_mask,
    test_mask,
    correlation_threshold=0.8,
)

prepared_b = prepare_feature_subset(
    subset_b,
    development_mask,
    test_mask,
    correlation_threshold=0.8,
)

In [5]:
with open(
    FEATURE_SELECTION_FILE,
    "r",
    encoding="utf-8",
) as file:
    feature_selection = json.load(file)

selected_a = feature_selection[
    "subset_a"
]["selected_features"]

selected_b = feature_selection[
    "subset_b"
]["selected_features"]

print(
    f"Subset A selected features: "
    f"{len(selected_a)}"
)

print(
    f"Subset B selected features: "
    f"{len(selected_b)}"
)

Subset A selected features: 6
Subset B selected features: 24


In [6]:
def select_columns(
    prepared_subset: dict,
    selected_features: list[str],
):
    indices = [
        prepared_subset[
            "features"
        ].index(feature)
        for feature
        in selected_features
    ]

    return (
        prepared_subset[
            "X_development"
        ][:, indices],
        prepared_subset[
            "X_test"
        ][:, indices],
    )

In [7]:
X_dev_a, X_test_a = select_columns(
    prepared_a,
    selected_a,
)

X_dev_b, X_test_b = select_columns(
    prepared_b,
    selected_b,
)

print(
    f"Subset A — development: "
    f"{X_dev_a.shape}"
)

print(
    f"Subset A — test: "
    f"{X_test_a.shape}"
)

print(
    f"Subset B — development: "
    f"{X_dev_b.shape}"
)

print(
    f"Subset B — test: "
    f"{X_test_b.shape}"
)

Subset A — development: (4741, 6)
Subset A — test: (365, 6)
Subset B — development: (4741, 24)
Subset B — test: (365, 24)


## 2. Temporal validation strategies

Conventional random cross-validation is inappropriate for this time-ordered dataset because validation observations must occur chronologically after training observations.

Two complementary validation schemes are therefore used:

### Sliding window

Two consecutive years are used for training and the following two years for validation. Both windows advance by one year at each fold.

### Expanding window

Training begins with three years and expands by two additional years at each fold. The subsequent two years are always used for validation.

In [8]:
sliding_splits = get_year_based_splits(
    development_dates,
    strategy="sliding",
)

expanding_splits = get_year_based_splits(
    development_dates,
    strategy="expanding",
)

In [9]:
sliding_summary = (
    summarize_temporal_splits(
        development_dates,
        sliding_splits,
    )
)

sliding_summary

,Fold,Training,Validation,N_train,N_validation
0,1,2010–2011,2012–2013,723,731
1,2,2011–2012,2013–2014,731,730
2,3,2012–2013,2014–2015,731,730
3,4,2013–2014,2015–2016,730,731
4,5,2014–2015,2016–2017,730,731
5,6,2015–2016,2017–2018,731,730
6,7,2016–2017,2018–2019,731,730
7,8,2017–2018,2019–2020,730,731
8,9,2018–2019,2020–2021,730,731
9,10,2019–2020,2021–2022,731,730


In [10]:
expanding_summary = (
    summarize_temporal_splits(
        development_dates,
        expanding_splits,
    )
)

expanding_summary

,Fold,Training,Validation,N_train,N_validation
0,1,2010–2012,2013–2014,1089,730
1,2,2010–2014,2015–2016,1819,731
2,3,2010–2016,2017–2018,2550,730
3,4,2010–2018,2019–2020,3280,731
4,5,2010–2020,2021–2022,4011,730


## 3. Model configurations

The final hyperparameter configurations identified in the original study are evaluated using the same temporal cross-validation strategies.

Each combination of:

- validation strategy;
- predictor subset;
- model family;

defines one independent configuration.

This produces 12 configurations in total.

In [12]:
MODEL_NAMES = [
    "LightGBM",
    "CatBoost",
    "MLP",
]

SUBSETS = {
    "a": {
        "X": X_dev_a,
        "features": selected_a,
    },
    "b": {
        "X": X_dev_b,
        "features": selected_b,
    },
}

SPLITS = {
    "sliding": sliding_splits,
    "expanding": expanding_splits,
}

## 4. Temporal cross-validation performance

Each final hyperparameter configuration is evaluated using the corresponding temporal validation folds.

The mean AUC-ROC across folds is used to compare configurations.

In [15]:
cv_results = []

for strategy in [
    "sliding",
    "expanding",
]:
    for model_name in MODEL_NAMES:
        for subset_name in [
            "a",
            "b",
        ]:
            result = evaluate_temporal_cv(
                model_name=model_name,
                X=SUBSETS[
                    subset_name
                ]["X"],
                y=y_development,
                splits=SPLITS[
                    strategy
                ],
                params=OPTIMAL_PARAMS[
                    strategy
                ][
                    model_name
                ][
                    subset_name
                ],
            )

            cv_results.append(
                {
                    "Strategy": strategy,
                    "Subset": (
                        subset_name.upper()
                    ),
                    "Model": model_name,
                    "Mean_AUC": result[
                        "mean_auc"
                    ],
                    "Fold_AUCs": result[
                        "fold_aucs"
                    ],
                }
            )

            print(
                f"{strategy:9s} | "
                f"{model_name:8s} | "
                f"Subset "
                f"{subset_name.upper()} | "
                f"AUC = "
                f"{result['mean_auc']:.4f}"
            )

sliding   | LightGBM | Subset A | AUC = 0.5615
sliding   | LightGBM | Subset B | AUC = 0.5740
sliding   | CatBoost | Subset A | AUC = 0.5689
sliding   | CatBoost | Subset B | AUC = 0.5807
sliding   | MLP      | Subset A | AUC = 0.5621
sliding   | MLP      | Subset B | AUC = 0.5758
expanding | LightGBM | Subset A | AUC = 0.5737
expanding | LightGBM | Subset B | AUC = 0.5816
expanding | CatBoost | Subset A | AUC = 0.5672
expanding | CatBoost | Subset B | AUC = 0.5870
expanding | MLP      | Subset A | AUC = 0.5750
expanding | MLP      | Subset B | AUC = 0.5828


In [16]:
cv_results_df = pd.DataFrame(
    cv_results
)

cv_results_df[
    [
        "Strategy",
        "Subset",
        "Model",
        "Mean_AUC",
    ]
].sort_values(
    "Mean_AUC",
    ascending=False,
)

,Strategy,Subset,Model,Mean_AUC
9,expanding,B,CatBoost,0.586963
11,expanding,B,MLP,0.582805
7,expanding,B,LightGBM,0.581595
3,sliding,B,CatBoost,0.580702
5,sliding,B,MLP,0.575838
10,expanding,A,MLP,0.575003
1,sliding,B,LightGBM,0.574046
6,expanding,A,LightGBM,0.573714
2,sliding,A,CatBoost,0.568910
8,expanding,A,CatBoost,0.567215


## 5. Cross-validation summary

The temporal cross-validation results reproduce the performance obtained in the original study.

Across both validation strategies, models using Subset B generally achieved higher mean AUC-ROC values than those using Subset A.

The highest mean validation performance was obtained by CatBoost using Subset B under the expanding-window strategy (AUC-ROC = 0.5870).

Overall, the validation AUC values remained modest, ranging approximately from 0.56 to 0.59, indicating limited but consistent predictive information in the environmental and temporal predictors.

In [17]:
cv_summary = (
    cv_results_df[
        [
            "Strategy",
            "Subset",
            "Model",
            "Mean_AUC",
        ]
    ]
    .sort_values(
        [
            "Strategy",
            "Subset",
            "Model",
        ]
    )
    .reset_index(drop=True)
)

cv_summary["Mean_AUC"] = (
    cv_summary["Mean_AUC"]
    .round(4)
)

cv_summary

,Strategy,Subset,Model,Mean_AUC
0,expanding,A,CatBoost,0.5672
1,expanding,A,LightGBM,0.5737
2,expanding,A,MLP,0.5750
3,expanding,B,CatBoost,0.5870
4,expanding,B,LightGBM,0.5816
5,expanding,B,MLP,0.5828
6,sliding,A,CatBoost,0.5689
7,sliding,A,LightGBM,0.5615
8,sliding,A,MLP,0.5621
9,sliding,B,CatBoost,0.5807


In [19]:
cv_summary.to_csv(
    CV_RESULTS_FILE,
    index=False,
)

print(
    f"✓ Temporal CV results saved to: "
    f"{CV_RESULTS_FILE}"
)

✓ Temporal CV results saved to: C:\Users\migue\Desktop\Healthcare-AI\copd-exacerbation-forecasting\reports\temporal_cv_results.csv


## 6. Modelling-stage conclusion

The complete modelling pipeline up to temporal validation has now been reconstructed.

The selected predictor sets and the final hyperparameter configurations reproduce the results of the original Bachelor Thesis. The independent 2023 test set has remained isolated throughout feature selection and temporal cross-validation.

The next stage evaluates the final models on this unseen test year and analyses their predictions using model-performance metrics, ROC curves, confusion matrices and SHAP explanations.